# Diabetic Retinopathy KFold Reproducibility

This notebook reruns and verifies the reviewer-facing diabetic retinopathy experiments:

- patient-level outer train/test split;
- patient-level 5-fold validation on the outer-train set;
- FKG-UM image, FKG-UM table, FKG-MM proposed fusion, and fusion strategy variants;
- full metric set: Sensitivity, Specificity, F1, AUC-ROC, and AUC-PR.

The FKG/FKGS step uses `ROOT_DATA/train_test_selection/train.csv` and the exact `train_kfold/fold_*/train.csv` / `val.csv` manifests created for the deep baselines. That keeps patient IDs grouped and makes the validation folds auditable.

Reviewer formula fixes included in this notebook:

- Tensor and Hadamard fusion fit train-only cross-SVD projections from `F_img.T @ F_tab`, avoiding the previous dimension ambiguity from flattened Kronecker features.
- Filter fusion uses an explicit `--filter-corr` threshold and rejects both intra-modal and inter-modal correlated candidates.
- Wrapper fusion evaluates the minimum initialized set before adding features and respects `max_img` / `max_tab` limits.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

NOTEBOOK_PATH = Path.cwd()
REPO_ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / 'Source_code').exists() else NOTEBOOK_PATH.parent
os.chdir(REPO_ROOT)
print('Repository:', REPO_ROOT)

venv_python = REPO_ROOT / '.venv_deep_baselines' / 'Scripts' / 'python.exe'
PYTHON = os.environ.get('PYTHON_EXE', str(venv_python if venv_python.exists() else Path(sys.executable)))
print('Python:', PYTHON)

In [ ]:
RUN_ID = os.environ.get('RUN_ID', '20260921')
RUN_TAG = os.environ.get('RUN_TAG', f'kfold_rerun_{RUN_ID}_formula_fix')
DEVICE = os.environ.get('DEVICE', 'auto')
RESNET_ARCH = os.environ.get('RESNET_ARCH', 'resnet50')
EPOCHS = int(os.environ.get('EPOCHS', '10'))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '16'))
FIS_BACKEND = os.environ.get('FIS_BACKEND', 'cpu')
FKG_BACKEND = os.environ.get('FKG_BACKEND', 'auto')
FKGS_RAN = os.environ.get('FKGS_RAN', '15 20').split()
FKGS_EPSILON = os.environ.get('FKGS_EPSILON', '0.2 0.3').split()
FKGS_TURNS = os.environ.get('FKGS_TURNS', '1')
FKGS_WORKERS = os.environ.get('FKGS_WORKERS', '1')
FKG_MODALITIES = os.environ.get(
    'FKG_MODALITIES',
    'table image fusion fusion_filter fusion_hadamard fusion_tensor fusion_wrapper',
).split()

SPLIT_ROOT = Path('ROOT_DATA/train_test_selection')
DEEP_RESULTS = SPLIT_ROOT / 'deep_baselines' / f'kfold_rerun_{RUN_ID}'
FKGS_OUTPUT = Path('Source_code/data/Dataset_diabetic') / f'KFold_feature_selection_rerun_{RUN_ID}'
FKGS_REPORT = Path('Source_code/data/result') / f'KFold_feature_selection_rerun_{RUN_ID}'
COMPARISON_STEM = Path('result') / f'diabetic_retinopathy_model_comparison_kfold_rerun_{RUN_ID}'

RESTRICT_IMAGE_IDS = SPLIT_ROOT / 'train.csv'
FOLD_MANIFEST_ROOT = SPLIT_ROOT / 'train_kfold'
PROTOCOL_TEXT = (
    'Patient-grouped 5-fold CV with shared deep-baseline train image IDs; '
    'fold_manifest_root=ROOT_DATA/train_test_selection/train_kfold; '
    'formula_fix cross-SVD fusion.'
)

print('RUN_ID:', RUN_ID)
print('RUN_TAG:', RUN_TAG)
print('Modalities:', FKG_MODALITIES)
print('Deep results:', DEEP_RESULTS)
print('FKG/FKGS report:', FKGS_REPORT)
print('Comparison stem:', COMPARISON_STEM)

In [ ]:
def run_command(command: list[str], *, skip_if: Path | None = None, force: bool = False) -> None:
    if skip_if is not None and skip_if.exists() and not force:
        print(f'[SKIP] {skip_if} already exists')
        return
    print('[RUN]', ' '.join(str(part) for part in command))
    subprocess.run([str(part) for part in command], check=True, cwd=REPO_ROOT)

def read_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

## 1. Create Patient-Aware Splits

This cell is fast. It writes `ROOT_DATA/train_test_selection/summary.json`, `train.csv`, `test.csv`, and `train_kfold/fold_*` manifests. The assertions below fail if any patient appears on both sides of a split.

In [ ]:
run_command([
    PYTHON,
    'Source_code/main/diabetic_retinopathy/create_root_data_image_train_test_split.py',
    '--root-data', 'ROOT_DATA',
    '--image-dir', 'ROOT_DATA/fundus_photos_224',
    '--materialize', 'none',
    '--path-mode', 'relative',
    '--overwrite',
])

In [ ]:
split_summary = read_json(SPLIT_ROOT / 'summary.json')
assert split_summary['patient_overlap_count'] == 0, split_summary['patient_overlap_count']
for fold in split_summary['train_kfold']['folds']:
    assert fold['patient_overlap_count'] == 0, fold
print('Outer split patients:', split_summary['train']['patients'], '/', split_summary['test']['patients'])
print('Outer split images:', split_summary['train']['images'], '/', split_summary['test']['images'])
pd.DataFrame(split_summary['train_kfold']['folds'])[['fold', 'patient_overlap_count']]

## 2. Run Deep Baselines

This is the slow CNN/MLP step. It is skipped when `summary.csv` already exists for the chosen `RUN_ID`; delete the output folder or pass `force=True` to rerun.

In [ ]:
run_command([
    PYTHON,
    'Source_code/main/diabetic_retinopathy/run_deep_multimodal_baselines.py',
    '--split-root', str(SPLIT_ROOT),
    '--tabular-csv', 'Source_code/data/Dataset_diabetic/data_process.csv',
    '--results-dir', str(DEEP_RESULTS),
    '--models', 'all',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', DEVICE,
    '--resnet-arch', RESNET_ARCH,
], skip_if=DEEP_RESULTS / 'summary.csv')

## 3. Run FIS + Native FKG + FKGS

This cell uses the same outer-train images and the exact same fold manifests as the deep baseline. It covers FKG-UM image/table, FKG-MM proposed fusion, and the filter/hadamard/tensor/wrapper fusion strategies.

In [ ]:
fkg_command = [
    PYTHON,
    'Source_code/main/diabetic_retinopathy/Preprocess_kfold_feature_selection.py',
    '--modalities', *FKG_MODALITIES,
    '--run-fkgs',
    '--reuse-fkgs',
    '--run-fkg',
    '--fis-engine', 'native',
    '--native-backend', FIS_BACKEND,
    '--fkg-backend', FKG_BACKEND,
    '--restrict-image-ids', str(RESTRICT_IMAGE_IDS),
    '--fold-manifest-root', str(FOLD_MANIFEST_ROOT),
    '--run-tag', RUN_TAG,
    '--ran', *FKGS_RAN,
    '--e', *FKGS_EPSILON,
    '--fkgs-turns', FKGS_TURNS,
    '--fkgs-workers', FKGS_WORKERS,
    '--output-root', str(FKGS_OUTPUT.relative_to('Source_code')),
    '--report-root', str(FKGS_REPORT.relative_to('Source_code')),
]
run_command(fkg_command, skip_if=FKGS_REPORT / 'kfold_modality_mean_std_summary.csv')

## 4. Build Comparison Table

The output CSV contains the full metric columns; the Markdown file is a compact reviewer table.

In [ ]:
run_command([
    PYTHON,
    'Source_code/main/diabetic_retinopathy/collect_kfold_model_comparison.py',
    '--deep-summary', str(DEEP_RESULTS / 'summary.csv'),
    '--deep-config', str(DEEP_RESULTS / 'config.json'),
    '--fkgs-summary', str(FKGS_REPORT / 'kfold_fkgs_mean_std_summary.csv'),
    '--fkgs-tables', str(FKGS_REPORT / 'kfold_fkgs_tables.csv'),
    '--fkg-summary', str(FKGS_REPORT / 'kfold_modality_mean_std_summary.csv'),
    '--output-stem', str(COMPARISON_STEM),
    '--protocol', PROTOCOL_TEXT,
], skip_if=COMPARISON_STEM.with_suffix('.csv'))

In [ ]:
comparison_csv = COMPARISON_STEM.with_suffix('.csv')
comparison_md = COMPARISON_STEM.with_suffix('.md')
comparison = pd.read_csv(comparison_csv)
display_columns = [
    'model', 'data_type', 'source_family',
    'accuracy_pct', 'sensitivity_pct', 'specificity_pct',
    'precision_pct', 'f1_pct', 'auc_roc_pct', 'auc_pr_pct',
]
display(comparison[display_columns])
print('CSV:', comparison_csv.resolve())
print('Markdown:', comparison_md.resolve())

## 5. Audit Fold Artifacts

The cells below are non-training checks for reviewers. They verify that the FKG summary rows came from patient-level folds with zero overlap and that the full metric columns are present.

In [ ]:
run_summary = pd.read_csv(FKGS_REPORT / 'kfold_run_summary.csv')
assert (run_summary['patient_overlap_count'].fillna(0).astype(int) == 0).all()
required_columns = ['fkg_sensitivity', 'fkg_specificity', 'fkg_f1', 'fkg_auc_roc', 'fkg_auc_pr']
missing = [column for column in required_columns if column not in run_summary.columns]
assert not missing, missing
print('FKG/FKGS fold rows:', len(run_summary))
print('Modalities:', sorted(run_summary['modality'].dropna().unique()))
run_summary[['modality', 'fold', 'splitter', 'patient_overlap_count']].head(10)